# Solving for LF Accuracies with PyTorch Gradient Descent

This notebook shows how Snorkel estimates labeling function accuracies from agreement patterns using gradient descent.

**The Problem:**
- We have 3 labeling functions (LFs)
- We observe how often they agree with each other
- We want to find each LF's accuracy (α)

**The Key Equation:**
$$P(\text{LF}_i \text{ and } \text{LF}_j \text{ agree}) = \alpha_i \alpha_j + (1-\alpha_i)(1-\alpha_j)$$

(They agree when both are correct OR both are wrong)

In [ ]:
import torch
import matplotlib.pyplot as plt

## Step 1: Define the Observed Agreement Rates

From our 1000 movie reviews, we counted how often each pair of LFs agreed:

In [ ]:
# Observed agreement rates between LF pairs
observed_agreements = {
    ('LF1', 'LF2'): 0.85,  # LF1 and LF2 agree 85% of the time
    ('LF1', 'LF3'): 0.80,  # LF1 and LF3 agree 80% of the time  
    ('LF2', 'LF3'): 0.90,  # LF2 and LF3 agree 90% of the time
}

# Convert to tensor
obs = torch.tensor([0.85, 0.80, 0.90])

print("Observed Agreement Rates:")
for pair, rate in observed_agreements.items():
    print(f"  {pair[0]} & {pair[1]}: {rate:.0%}")

## Step 2: Define α as Learnable Parameters

We'll use gradient descent to find the α values that best explain the observed agreements.

In [ ]:
# Initialize α's randomly (between 0.5 and 1.0, since accuracy > 50%)
torch.manual_seed(42)
alpha = torch.tensor([0.6, 0.6, 0.6], requires_grad=True)

print(f"Initial α values: α1={alpha[0]:.3f}, α2={alpha[1]:.3f}, α3={alpha[2]:.3f}")

## Step 3: Define the Agreement Function

$$P(\text{agree}) = \alpha_i \cdot \alpha_j + (1-\alpha_i) \cdot (1-\alpha_j)$$

In [ ]:
def predict_agreement(alpha):
    """Predict agreement rates given α values."""
    a1, a2, a3 = alpha[0], alpha[1], alpha[2]
    
    # Agreement = both correct + both wrong
    agree_12 = a1 * a2 + (1 - a1) * (1 - a2)
    agree_13 = a1 * a3 + (1 - a1) * (1 - a3)
    agree_23 = a2 * a3 + (1 - a2) * (1 - a3)
    
    return torch.stack([agree_12, agree_13, agree_23])

# Test with initial values
pred = predict_agreement(alpha)
print(f"Predicted agreements with initial α: {pred.detach().numpy()}")
print(f"Observed agreements:                  {obs.numpy()}")

## Step 4: Gradient Descent!

Minimize the squared error between predicted and observed agreements.

In [ ]:
# Re-initialize
alpha = torch.tensor([0.6, 0.6, 0.6], requires_grad=True)

# Hyperparameters
learning_rate = 0.5
n_iterations = 100

# Track history for plotting
history = {'loss': [], 'alpha1': [], 'alpha2': [], 'alpha3': []}

print("Gradient Descent Progress:")
print("=" * 60)

for i in range(n_iterations):
    # Forward pass: predict agreement rates
    pred = predict_agreement(alpha)
    
    # Compute loss (MSE)
    loss = ((pred - obs) ** 2).sum()
    
    # Backward pass: compute gradients
    loss.backward()
    
    # Update α's (gradient descent step)
    with torch.no_grad():
        alpha -= learning_rate * alpha.grad
        
        # Clamp to valid range [0.5, 1.0]
        alpha.clamp_(0.5, 1.0)
    
    # Zero gradients for next iteration
    alpha.grad.zero_()
    
    # Record history
    history['loss'].append(loss.item())
    history['alpha1'].append(alpha[0].item())
    history['alpha2'].append(alpha[1].item())
    history['alpha3'].append(alpha[2].item())
    
    # Print progress
    if i < 10 or (i + 1) % 10 == 0:
        print(f"Iter {i+1:3d}: α1={alpha[0]:.4f}, α2={alpha[1]:.4f}, α3={alpha[2]:.4f}, loss={loss:.6f}")

## Step 5: Visualize Convergence

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot loss
axes[0].plot(history['loss'], 'b-', linewidth=2)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Loss Convergence')
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

# Plot α values
axes[1].plot(history['alpha1'], 'r-', label='α₁', linewidth=2)
axes[1].plot(history['alpha2'], 'g-', label='α₂', linewidth=2)
axes[1].plot(history['alpha3'], 'b-', label='α₃', linewidth=2)
axes[1].axhline(y=0.80, color='r', linestyle='--', alpha=0.5, label='α₁ target')
axes[1].axhline(y=0.90, color='g', linestyle='--', alpha=0.5, label='α₂ target')
axes[1].axhline(y=0.85, color='b', linestyle='--', alpha=0.5, label='α₃ target')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('α value')
axes[1].set_title('α Convergence')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 6: Verify the Solution

In [ ]:
print("\n" + "=" * 60)
print("FINAL SOLUTION")
print("=" * 60)

print(f"\nEstimated Accuracies:")
print(f"  α₁ = {alpha[0]:.4f}  (LF₁ is correct {alpha[0]*100:.1f}% of the time)")
print(f"  α₂ = {alpha[1]:.4f}  (LF₂ is correct {alpha[1]*100:.1f}% of the time)")
print(f"  α₃ = {alpha[2]:.4f}  (LF₃ is correct {alpha[2]*100:.1f}% of the time)")

# Verify
pred = predict_agreement(alpha)
print(f"\nVerification:")
print(f"  {'Pair':<12} {'Observed':>10} {'Predicted':>10} {'Error':>10}")
print(f"  {'-'*44}")
pairs = [('LF₁ & LF₂', 0), ('LF₁ & LF₃', 1), ('LF₂ & LF₃', 2)]
for name, idx in pairs:
    error = abs(pred[idx].item() - obs[idx].item())
    print(f"  {name:<12} {obs[idx].item():>10.4f} {pred[idx].item():>10.4f} {error:>10.6f}")

## Key Takeaways

1. **We estimated LF accuracies without knowing true labels!**
   - Only used agreement patterns between LFs

2. **Gradient descent finds α values that explain observed agreements**
   - Loss = how well predicted agreements match observed
   - Minimize loss → find correct α's

3. **This is the core of Snorkel's Label Model**
   - Real Snorkel uses more sophisticated probabilistic modeling
   - But the intuition is the same!